# Museum Guide — RAG v2 (with an LLM-as-a-Judge evaluation)

A tiny Retrieval-Augmented Generation assistant for the **Raclin Murphy Museum of Art**, built on the exact Week 8 pattern: embed your documents into a vector database, search it for the passages closest to a question, and hand those passages to the model to answer with citations.

The only thing that changed from the Week 8 code is the **source of the documents**: instead of copying a pre-built database from Drive, we build one from your spreadsheet. Run the cells top to bottom.

**Before you start:** add your `OPENAI_API_KEY` in Colab Secrets (the key icon on the left).

> **v2 changes:** generator model is now `gpt-5.4-mini`; all six spreadsheet tabs are loaded (Artworks, Galleries, Exhibitions, Floor Plan Directory, Wayfinding, Themes); retrieval `k` is 6; and **Step 5** adds an LLM-as-a-Judge that scores the guide's answers to the 30 test questions.

## Step 0: Setup

One install cell (LangChain, its OpenAI and Chroma connectors, Chroma itself, and `openpyxl` for reading the spreadsheet), at the versions we tested. If pip ends with a red `ERROR` about packages we never import, the install still worked.

In [1]:
# pinned: these exact versions were tested together on 2026-10-11
# (LangChain itself, its OpenAI and Chroma connectors, and the Chroma vector database; openpyxl reads .xlsx)
!pip install -q langchain==1.4.3 langchain-core==1.6.6 langchain-openai==1.6.6 langchain-chroma==1.1.0 chromadb==1.5.9 openpyxl

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.


In [3]:
import os
import pandas as pd
from google.colab import userdata

# Your OpenAI key lives in Colab Secrets (the key icon). LangChain looks for it in this environment variable.
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

# The LangChain pieces we use:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings          # OpenAI's chat model and embedding model
from langchain_chroma import Chroma                                # Chroma, our vector database
from langchain_core.documents import Document                      # one record = one Document
from langchain_core.messages import SystemMessage, HumanMessage    # the message types in a chat

# temperature=0 means the model picks its most likely words, so everyone's output is close to the same.
llm = ChatOpenAI(model="gpt-5.4-mini", temperature=0)                 # writes the replies
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")        # turns text into vectors for search

## Step 1: Upload your spreadsheet

Run this cell and pick `Raclin_Murphy_Museum_Galleries_Dataset.xlsx` from your computer. It lands in this Colab machine so the next cell can read it. (You re-upload each time Colab restarts; to skip that, mount Google Drive instead and point `SPREADSHEET` below at the file's Drive path.)

In [4]:
from google.colab import files
uploaded = files.upload()      # click "Choose Files" and select the .xlsx

Saving Raclin_Murphy_Museum_Galleries_Dataset.xlsx to Raclin_Murphy_Museum_Galleries_Dataset.xlsx


## Step 2: Build the vector database from the spreadsheet

This is the one new piece. We turn **each row into one Document**: the text the model reads, plus a `title` and `locator` for citations. Then we embed them all into a fresh Chroma database. Each sheet has two banner rows and a blank row before its real column headers, which is why `header=3`.

To point this notebook at different data later, change `SPREADSHEET` and the `SHEETS` list — nothing else.

In [5]:
SPREADSHEET = "Raclin_Murphy_Museum_Galleries_Dataset.xlsx"   # the file you uploaded above

# Which sheets to load, and which columns to use as the (title, locator) label the model cites.
SHEETS = [
    {"sheet": "Artworks",             "header": 3, "title_col": "Title",                  "locator_cols": ["Gallery (assigned)", "Accession No."]},
    {"sheet": "Galleries",            "header": 3, "title_col": "Gallery",                 "locator_cols": ["Level"]},
    {"sheet": "Exhibitions",          "header": 3, "title_col": "Title",                  "locator_cols": ["Status", "Open", "Close"]},
    {"sheet": "Floor Plan Directory", "header": 3, "title_col": "Space",                  "locator_cols": ["Floor", "Position on floor"]},
    {"sheet": "Wayfinding",           "header": 3, "title_col": "Subject",                "locator_cols": ["Type"]},
    {"sheet": "Themes (Marble portfolios)", "header": 3, "title_col": "Theme / Sub-collection", "locator_cols": ["Gallery"]},
]

def row_to_text(row, cols):
    # one record -> one readable passage: every non-empty "Column: value" line
    return "\n".join(f"{c}: {row[c]}" for c in cols
                     if c != "#" and pd.notna(row[c]) and str(row[c]).strip())

docs = []
for spec in SHEETS:
    df = pd.read_excel(SPREADSHEET, sheet_name=spec["sheet"], header=spec["header"]).dropna(how="all")
    cols = list(df.columns)
    for _, row in df.iterrows():
        title = str(row.get(spec["title_col"], "")).strip()
        if not title or title.lower() == "nan":               # skip blank rows
            continue
        locator = ", ".join(str(row[c]).strip() for c in spec["locator_cols"]
                            if c in df.columns and pd.notna(row[c]) and str(row[c]).strip())
        docs.append(Document(page_content=row_to_text(row, cols),
                             metadata={"title": title, "locator": locator or spec["sheet"]}))

print(f"Built {len(docs)} records from {SPREADSHEET}.")

# Embed every record into a fresh Chroma database. No persist_directory, so it lives in memory and is
# rebuilt each run; add persist_directory="/content/museum_db" to save it between runs.
collection = Chroma.from_documents(docs, embedding=embeddings, collection_name="museum")
print(f"collection database: {collection._collection.count():,} records ready to search")

Built 895 records from Raclin_Murphy_Museum_Galleries_Dataset.xlsx.
collection database: 895 records ready to search


## Step 3: Search and answer (the RAG core)

The same pattern as Week 8's `search_readings`, pointed at the museum: `search_collection` finds the 4 records closest in meaning to a question and labels each with its title and locator, so every answer can carry citations. Then one `llm.invoke` call stitches together the instructions, the retrieved records, and the question.

In [6]:
def search_collection(question, k=6):   # k raised from 4 -> 6 to improve recall on narrow queries
    # Embed the question and find the k records closest to it in meaning (the "R" in RAG).
    results = collection.similarity_search(question, k=k)
    # Label every record with its title and locator so the model can cite it. Records are separated by blank lines.
    return "\n\n".join(f"[{d.metadata['title']}, {d.metadata['locator']}] {d.page_content}" for d in results)

# The system message is the assistant's standing instructions. It goes first in every call.
SYSTEM = SystemMessage(
    "You are a gallery guide for the Raclin Murphy Museum of Art at the University of Notre Dame. Answer using "
    "only the collection records you are given, and cite them as (title, locator). If the records don't cover "
    "it, say so. Keep replies to three sentences.")

# 1. The question.
question = "What African artworks are in the collection?"
# 2. One call: instructions + retrieved records + the question, as ONE list of messages.
reply = llm.invoke([SYSTEM, SystemMessage("Collection records:\n" + search_collection(question)), HumanMessage(question)])
print(reply.content)                 # 3. the answer, with citations

The records only identify a few African artworks in the gallery: an Egungun Masquerade Costume, Yinka Shonibare’s *Earth Kid (Boy)*, a rare Model Boat from Cameroon, and an ancient Egyptian carving of Cleopatra (African Art, Level 1). One specific item record is also listed: *Untitled* (2017.025.540), but the record does not provide a description of the work itself (Untitled, African Art, 2017.025.540). If you want, I can also tell you where the African Art gallery is located in the museum (African Art, Location).


## Step 4: Ask your own questions

`ask()` wraps the same three-part call so you can fire off questions in a loop. Edit the list, or call `ask("...")` yourself.

In [7]:
def ask(question):
    reply = llm.invoke([SYSTEM, SystemMessage("Collection records:\n" + search_collection(question)), HumanMessage(question)])
    return reply.content

for q in ["What did Zhang Huan use to create his 1965 Last Supper artwork?",
          "Who created Wall of Light Black in 1998?",
          "What colors are used in the painting Spring by Natalia Goncharova?",
          "What artifacts are used in the White Rit sculture created by Vanessa German?",
          "In what year did Francesco de Mura create Bacchus and Ceres?",
          "Who gifted the 1786 painting Marguerite Baudard de Saint James to the Raclin Murphy?",
          "Why is The Artist’s Studio, a Sketch by Jean-Léon Gérôme an important piece in the Raclin Murphy Museum?",
          "Have any pieces of art been donated as past of the Bernard Norling and Mary T. Norling Endowment for 18th- and 19th-Century Sculpture?"]:
    print("YOU: ", q)
    print("GUIDE:", ask(q), "\n")

YOU:  What did Zhang Huan use to create his 1965 Last Supper artwork?
GUIDE: Zhang Huan made *Ash Banquet #4* with paint mixed with ash collected from burnt incense sticks at Buddhist temples in Shanghai, so the work is literally formed from the residue of countless prayers (Ash Banquet #4, 2021.012). It is a monumental gray reinterpretation of Leonardo da Vinci’s *The Last Supper* showing Christ and the twelve apostles (Ash Banquet #4, 2021.012). 

YOU:  Who created Wall of Light Black in 1998?
GUIDE: Wall of Light Black was created by Sean Scully (American, b. 1945) in 1998. It is listed as a painting in the International Modern and Contemporary Art gallery (Wall of Light Black, 1999.027). 

YOU:  What colors are used in the painting Spring by Natalia Goncharova?
GUIDE: The record for *Spring* by Natalia Goncharova does not specify the palette, so the exact colors are not documented there (Spring, 1994.027.007). It is described as an avant-garde composition and part of her treatment 

In [8]:
def ask(question):
    reply = llm.invoke([SYSTEM, SystemMessage("Collection records:\n" + search_collection(question)), HumanMessage(question)])
    return reply.content

for q in ["Where can I find the chapel in the Raclin Murphy?",
          "How do I get to Art Studio B in the Raclin Murphy Museum from the Welcome Desk?",
          "What are the current temporary exhibitions at the Raclin Murphy Museum?",
          "When will Mimmo Paladino: Present into Past be at the Raclin Murphy Museum until?",
          "How big is Mimmo Paladino: Present into Past?",
          "What upcoming exhibition is coming to Raclin Murphy in February?",
          "Has the Raclin Murphy Museum ever had an Emilio Sanchez exhibit?",]:
    print("YOU: ", q)
    print("GUIDE:", ask(q), "\n")

YOU:  Where can I find the chapel in the Raclin Murphy?
GUIDE: The Mary, Queen of Families Chapel is on Level 2 at the south-center of the museum, between the two wings of European Art Through 1700 and across the balcony from the Founders Room. From the Level 1 entrance, cross the Grand Atrium and take the elevator or the North or South stairs up one level to Level 2; the chapel is straight across the balcony on the south side. (Where is the chapel?, FAQ) 

YOU:  How do I get to Art Studio B in the Raclin Murphy Museum from the Welcome Desk?
GUIDE: From the Welcome Desk on Level 1, go to the Grand Atrium and take the South stairs or the elevator up to Level 3. On Level 3, head to the south side off the central landing, east of Art Studio A; Art Studio B is there (Welcome Desk to Art Studio B, Route). 

YOU:  What are the current temporary exhibitions at the Raclin Murphy Museum?
GUIDE: The current temporary exhibition in the records is *Mimmo Paladino: Present into Past* (Mimmo Paladin

## Step 5: Evaluate with an LLM-as-a-Judge (anchored rubric)

This reuses the **anchored rubric** from Week 7 (`AAI_Week_7_2_FINAL`, Step 3) verbatim — the same `Graded` / `AnchoredScores` form and the same judge prompt — because that version spreads scores out instead of hitting the 5/5 ceiling. The only adaptation is wiring it to this museum guide: for each test question we retrieve the passages (`search_collection`), write an answer (`ask`), and hand the judge the question, the reference (correct) answer, the passages, and the answer. The judge runs on `gpt-5.4-mini`.

In [9]:
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate

# ----- exact cookie-cutter judge from Week 7, Step 3 (anchored rubric) -----
class Graded(BaseModel):
    reasoning: str = Field(description="check the answer against the scale first, in one or two sentences")
    score: int = Field(ge=1, le=5)

class AnchoredScores(BaseModel):                 # the judge fills this in, top to bottom: evidence first, then scores
    key_points: list[str] = Field(description="the key points in the reference answer, one per item")
    missing_points: list[str] = Field(description="key points the answer does not state")
    fluency: Graded
    correctness: Graded
    completeness: Graded
    groundedness: Graded

anchored_prompt = ChatPromptTemplate.from_template("""You are grading an answer written by an AI assistant. First list the key points in the reference answer and which of them the answer leaves out. Then score each aspect using its scale. Give your reasoning before each score.

Fluency
5: clear, well organized, easy to read. 3: understandable but wordy or poorly organized. 1: hard to follow.
Correctness (compare with the reference answer)
5: nothing false and nothing that contradicts the reference. 3: one significant error, or the main claim differs from the reference. 1: mostly wrong.
Completeness (use your list of key points)
5: states every key point. 4: misses one minor point. 3: misses one key point. 2: misses most key points. 1: states none of them.
Groundedness (compare with the passages)
5: every claim can be traced to a passage. 3: several claims are not in the passages. 1: mostly unsupported.

Question: {question}
Reference answer: {reference}
Passages the assistant was given:
{passages}

Answer to grade:
{answer}""")
anchored_judge = anchored_prompt | ChatOpenAI(model="gpt-5.4-mini", temperature=0).with_structured_output(AnchoredScores)
print("judge ready (gpt-5.4-mini, anchored rubric)")

judge ready (gpt-5.4-mini, anchored rubric)


### The 30 test questions

The same 15 from your original sheet plus 15 new ones, each with its reference (correct) answer. Two are intentionally things the dataset does not yet contain — the exact palette of *Spring* and the size of the Paladino show — so the judge will flag them as the remaining gaps rather than hiding them.

In [10]:
TESTS = [
    {"question": "What did Zhang Huan use to create his 1965 Last Supper artwork?", "reference": "Paint with Ash on Linen"},
    {"question": "Who created Wall of Light Black in 1998?", "reference": "Sean Scully"},
    {"question": "What colors are used in the painting Spring by Natalia Goncharova?", "reference": "shades of blue, red/pink, gray, and earth colors"},
    {"question": "What artifacts are used in the White Rit sculture created by Vanessa German?", "reference": "Beads, fabric, nails, paint, baby doll legs, and a toy horse on wooden Rit Dye box"},
    {"question": "In what year did Francesco de Mura create Bacchus and Ceres?", "reference": "Circa 1763"},
    {"question": "Who gifted the 1786 painting Marguerite Baudard de Saint James to the Raclin Murphy?", "reference": "Mrs. Fred J. Fisher"},
    {"question": "Why is The Artist\u2019s Studio, a Sketch by Jean-L\u00e9on G\u00e9r\u00f4me an important piece in the Raclin Murphy Museum?", "reference": "Open Ended"},
    {"question": "Have any pieces of art been donated as past of the Bernard Norling and Mary T. Norling Endowment for 18th- and 19th-Century Sculpture?", "reference": "Priestess at the Festival of Beans by Charles Henri-Joseph Cordier"},
    {"question": "Where can I find the chapel in the Raclin Murphy?", "reference": "Level 2"},
    {"question": "How do I get to Art Studio B in the Raclin Murphy Museum from the Welcome Desk?", "reference": "Use the south stairs to go up to Level 3. Make a left and enter Studio B."},
    {"question": "What are the current temporary exhibitions at the Raclin Murphy Museum?", "reference": "Mimmo Paladino: Present into Past"},
    {"question": "When will Mimmo Paladino: Present into Past be at the Raclin Murphy Museum until?", "reference": "December 13, 2026"},
    {"question": "How big is Mimmo Paladino: Present into Past?", "reference": "85.83 x 38.19 x 23.23 in."},
    {"question": "What upcoming exhibition is coming to Raclin Murphy in February?", "reference": "More Than: Artists in Collaboration for a Greater Good"},
    {"question": "Has the Raclin Murphy Museum ever had an Emilio Sanchez exhibit?", "reference": "Yes, from Nov 4, 2021 - Dec 18, 2021"},
    {"question": "What materials make up the Egungun Masquerade Costume?", "reference": "Cloth, wood, paint, indigo dye, cotton yarn, and feathers"},
    {"question": "Who created the sculpture Earth Kid (Boy)?", "reference": "Yinka Shonibare, CBE"},
    {"question": "In what year was Earth Kid (Boy) made?", "reference": "2020"},
    {"question": "Which endowment funded the acquisition of Earth Kid (Boy)?", "reference": "Walter R. Beardsley Endowment for Contemporary Art"},
    {"question": "Which culture created the Royal Mukyeem Elephant Mask?", "reference": "Kuba"},
    {"question": "What is the Bas-Relief of Queen Cleopatra made of?", "reference": "Travertine marble"},
    {"question": "On what level is the International Modern and Contemporary Art gallery located?", "reference": "Lower Level"},
    {"question": "Where is the Sculpture Court located?", "reference": "Lower Level, north end (next to International Modern and Contemporary Art)"},
    {"question": "What is the curatorial theme of the European and American Art 1700\u20131900 gallery?", "reference": "Time : Nature"},
    {"question": "Who donated the 1729 painting Our Lady of Guadalupe?", "reference": "The Ignacio Aranguren family (Gift of Mr. Ignacio Aranguren, ND '52, and family)"},
    {"question": "When does the exhibition \"Spectacles of Power and Faith: Colonial South American Art from the Thoma Foundation\" open?", "reference": "August 24, 2027"},
    {"question": "What 2025 exhibition featured Walter Osborne's portraits of Dublin?", "reference": "Homecoming: Walter Osborne's Portraits of Dublin, 1880\u20131900"},
    {"question": "Who created Tracery, and what is it made of?", "reference": "Deborah Butterfield; cast bronze"},
    {"question": "How do I get from African Art to the Mary, Queen of Families Chapel?", "reference": "From African Art on Level 1, cross the Grand Atrium and take the stairs or elevator up to Level 2; the chapel is on the south side between the two European Art Through 1700 wings."},
    {"question": "Who is the artist of Wisgak Gokbenagen (Black Ash Basket), and what is it made of?", "reference": "Christine Marie Rapp-Morseau; black ash and sweet grass"},
]

### Run the guide + judge over all 30, and collect the scores

For every question: retrieve, answer, then grade. We treat **correctness ≥ 4** as the LLM-judge's pass/fail, which is the column you'll paste back into the question sheet. (This makes 60 API calls, so it takes a minute.)

In [11]:
import pandas as pd

rows=[]
for i, t in enumerate(TESTS, 1):
    passages = search_collection(t["question"])        # k=6 from Step 3
    answer   = ask(t["question"])
    g = anchored_judge.invoke({"question": t["question"], "reference": t["reference"],
                               "passages": passages, "answer": answer})
    rows.append({
        "Question": t["question"],
        "RAG 2 Answer": answer,
        "Correct Answer": t["reference"],
        "Correctness": g.correctness.score,
        "Completeness": g.completeness.score,
        "Groundedness": g.groundedness.score,
        "Fluency": g.fluency.score,
        "LLM As A Judge": g.correctness.score >= 4,          # pass/fail used to grade the sheet
        "Judge reasoning (correctness)": g.correctness.reasoning,
    })
    print(f"{i:2d}. correctness {g.correctness.score}/5  |  {t['question'][:60]}")

results_df = pd.DataFrame(rows)
passed = int(results_df["LLM As A Judge"].sum())
print(f"\nLLM-as-judge: {passed}/{len(results_df)} passed (correctness >= 4). "
      f"Mean correctness {results_df['Correctness'].mean():.2f}/5.")
results_df.to_csv("rag2_llm_judge_results.csv", index=False)   # download from the Files panel if you want the file
pd.set_option("display.max_colwidth", 70)
results_df[["Question","Correctness","Completeness","Groundedness","Fluency","LLM As A Judge"]]

 1. correctness 5/5  |  What did Zhang Huan use to create his 1965 Last Supper artwo
 2. correctness 5/5  |  Who created Wall of Light Black in 1998?
 3. correctness 1/5  |  What colors are used in the painting Spring by Natalia Gonch
 4. correctness 5/5  |  What artifacts are used in the White Rit sculture created by
 5. correctness 5/5  |  In what year did Francesco de Mura create Bacchus and Ceres?
 6. correctness 5/5  |  Who gifted the 1786 painting Marguerite Baudard de Saint Jam
 7. correctness 5/5  |  Why is The Artist’s Studio, a Sketch by Jean-Léon Gérôme an 
 8. correctness 3/5  |  Have any pieces of art been donated as past of the Bernard N
 9. correctness 5/5  |  Where can I find the chapel in the Raclin Murphy?
10. correctness 5/5  |  How do I get to Art Studio B in the Raclin Murphy Museum fro
11. correctness 5/5  |  What are the current temporary exhibitions at the Raclin Mur
12. correctness 5/5  |  When will Mimmo Paladino: Present into Past be at the Raclin
13. correct

,Question,Correctness,Completeness,Groundedness,Fluency,LLM As A Judge
0,What did Zhang Huan use to create his 1965 Last Supper artwork?,5,5,5,5,True
1,Who created Wall of Light Black in 1998?,5,5,5,5,True
2,What colors are used in the painting Spring by Natalia Goncharova?,1,1,5,5,False
3,What artifacts are used in the White Rit sculture created by Vanes...,5,5,5,5,True
4,In what year did Francesco de Mura create Bacchus and Ceres?,5,5,5,5,True
5,Who gifted the 1786 painting Marguerite Baudard de Saint James to ...,5,5,5,5,True
6,"Why is The Artist’s Studio, a Sketch by Jean-Léon Gérôme an import...",5,5,5,5,True
7,Have any pieces of art been donated as past of the Bernard Norling...,3,3,5,5,False
8,Where can I find the chapel in the Raclin Murphy?,5,5,5,5,True
9,How do I get to Art Studio B in the Raclin Murphy Museum from the ...,5,5,5,5,True


### Paste back into the question sheet

Two single-column blocks, in the same 30-row order as your sheet: the guide's answers (the **RAG 2 Answers** column) and the judge's verdict (the **LLM As A Judge** column). Copy each block from the output and paste it into the matching column of the `RAG Tests 2` tab.

In [12]:
print("----- RAG 2 Answers (paste into the RAG 2 Answers column) -----")
for a in results_df["RAG 2 Answer"]:
    print(str(a).replace("\n", " "))

print("\n----- LLM As A Judge (paste into the LLM As A Judge column) -----")
for v in results_df["LLM As A Judge"]:
    print(v)

----- RAG 2 Answers (paste into the RAG 2 Answers column) -----
Zhang Huan made *Ash Banquet #4* with paint mixed with ash collected from burnt incense sticks at Buddhist temples in Shanghai, so the work is formed from the residue of prayers (Ash Banquet #4, 2021.012). The record also notes that it is a 2021 reinterpretation of Leonardo da Vinci’s *The Last Supper* showing Christ and the twelve apostles (Ash Banquet #4, 2021.012).
Wall of Light Black was created by Sean Scully (American, b. 1945) in 1998. (Wall of Light Black, 1999.027)
The record for *Spring* by Natalia Goncharova does not specify its palette or exact colors, so I can’t confirm which colors are used from the collection record alone (Spring, 1994.027.007). The note only says it is an avant-garde composition and that the exact colors are not documented there (Spring, 1994.027.007).
The White Rit sculpture combines beads, fabric, nails, paint, baby doll legs, and a toy horse, all arranged on a wooden Rit Dye box (White R